# Local Vecchia Advection Simulation Comparison

Notebook version of `simulation/sim_vecchia_advec_comparison_042826.py` for local runs without Amarel queueing.

Compares exactly two models:

- `Vecc_Std`: standard Vecchia with extra temporal neighbors for fairness
- `Vecc_Advec`: advection-aware conditioning neighbor

The simulated field is generated directly on the target regular grid by circulant embedding. The summary includes mean, median, sd, RMSE/RMSRE, and `p90 - p10` spread of estimates for each parameter.


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.fft

GEMS_TCO_PATH = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
if GEMS_TCO_PATH not in sys.path:
    sys.path.insert(0, GEMS_TCO_PATH)

from GEMS_TCO import configuration as config
from GEMS_TCO import kernels_vecchia
from GEMS_TCO import kernels_vecchia_advec
from GEMS_TCO import orderings as _orderings

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
DELTA_LAT_BASE = 0.044
DELTA_LON_BASE = 0.063
T_STEPS = 8

MODELS = ["Vecc_Std", "Vecc_Advec"]
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
P_COLS = ["sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est", "advec_lat_est", "advec_lon_est", "nugget_est"]

print(f"Device: {DEVICE}")


## Settings

For a quick smoke test, set `num_iters = 1` or shrink the domain. The defaults below match the current full-domain simulation setup.


In [ ]:
true_dict = {
    "sigmasq": 10.0,
    "range_lat": 0.3,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.08,
    "advec_lon": -0.126,
    "nugget": 2.5,
}

lat_range = [-3.0, 2.0]
lon_range = [121.0, 131.0]

v = 0.5
mm_cond_number = 100
nheads = 0
limit_a = 20
limit_b = 20
limit_c = 20
daily_stride = 2
advec_lon_offset = DELTA_LON_BASE * 2

num_iters = 60
init_noise = 0.7
lbfgs_steps = 5
lbfgs_eval = 20
lbfgs_hist = 10
seed = 42

result_tag = "local_vecchia_advec_full_domain_neg0126_n60"

rng = np.random.default_rng(seed)
torch.manual_seed(seed)

output_path = Path(config.mac_estimates_day_path)
output_path.mkdir(parents=True, exist_ok=True)
checkpoint_file = output_path / f"sim_vecchia_advec_{result_tag}_checkpoint.csv"
raw_file = output_path / f"sim_vecchia_advec_{result_tag}_raw.csv"
summary_file = output_path / f"sim_vecchia_advec_{result_tag}_summary.csv"

print(true_dict)
print(f"Output path: {output_path}")


In [ ]:
def get_covariance_on_grid(lx, ly, lt, params):
    params = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat = lx - params[4] * lt
    u_lon = ly - params[5] * lt
    dist = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    return (phi1 / phi2) * torch.exp(-dist * phi2)


def build_target_grid(lat_range, lon_range):
    lats = torch.arange(min(lat_range), max(lat_range) + 0.0001, DELTA_LAT_BASE, device=DEVICE, dtype=DTYPE)
    lons = torch.arange(lon_range[0], lon_range[1] + 0.0001, DELTA_LON_BASE, device=DEVICE, dtype=DTYPE)
    lats = torch.round(lats * 10000) / 10000
    lons = torch.round(lons * 10000) / 10000
    g_lat, g_lon = torch.meshgrid(lats, lons, indexing="ij")
    grid_coords = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
    return lats, lons, grid_coords


def generate_field_values(n_lat, n_lon, t_steps, params):
    cpu = torch.device("cpu")
    f32 = torch.float32
    px, py, pt = 2 * n_lat, 2 * n_lon, 2 * t_steps

    lx = torch.arange(px, device=cpu, dtype=f32) * DELTA_LAT_BASE
    lx[px // 2:] -= px * DELTA_LAT_BASE
    ly = torch.arange(py, device=cpu, dtype=f32) * DELTA_LON_BASE
    ly[py // 2:] -= py * DELTA_LON_BASE
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt

    params_cpu = params.detach().cpu().float()
    Lx, Ly, Lt = torch.meshgrid(lx, ly, lt, indexing="ij")
    C = get_covariance_on_grid(Lx, Ly, Lt, params_cpu)
    S = torch.fft.fftn(C)
    S_real = torch.clamp(S.real, min=0.0)
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(S_real) * noise).real[:n_lat, :n_lon, :t_steps]
    return field.to(device=DEVICE, dtype=DTYPE)


def assemble_reg_map(field, grid_coords, true_params, t_offset=21.0):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid = grid_coords.shape[0]
    field_flat = field.reshape(n_grid, field.shape[-1])
    reg_map = {}

    for t_idx in range(field.shape[-1]):
        key = f"t{t_idx}"
        t_val = float(t_offset + t_idx)
        dummy = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0

        rows = torch.zeros((n_grid, 11), device=DEVICE, dtype=DTYPE)
        rows[:, :2] = grid_coords
        rows[:, 2] = field_flat[:, t_idx] + torch.randn(n_grid, device=DEVICE, dtype=DTYPE) * nugget_std
        rows[:, 3] = t_val
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        reg_map[key] = rows.detach()
    return reg_map


def compute_grid_ordering(grid_coords, mm_cond_number):
    coords_np = grid_coords.detach().cpu().numpy()
    ord_mm = _orderings.maxmin_cpp(coords_np)
    nns = _orderings.find_nns_l2(locs=coords_np[ord_mm], max_nn=mm_cond_number)
    return ord_mm, nns


In [ ]:
def true_to_log_params(d):
    phi2 = 1.0 / d["range_lon"]
    phi1 = d["sigmasq"] * phi2
    phi3 = (d["range_lon"] / d["range_lat"]) ** 2
    phi4 = (d["range_lon"] / d["range_time"]) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), d["advec_lat"], d["advec_lon"], np.log(d["nugget"])]


def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }


def calculate_rmsre(out_params, truth, zero_thresh=0.01):
    est = backmap_params(out_params)
    keys = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
    e = np.array([est[k] for k in keys])
    t = np.array([truth[k] for k in keys])
    mask = np.abs(t) >= zero_thresh
    rmsre = float(np.sqrt(np.mean(((e[mask] - t[mask]) / np.abs(t[mask])) ** 2)))
    mae_zero = float(np.mean(np.abs(e[~mask] - t[~mask]))) if (~mask).any() else float("nan")
    return rmsre, mae_zero, est


def make_random_init(rng, true_log, noise):
    noisy = list(true_log)
    for i in [0, 1, 2, 3, 6]:
        noisy[i] = true_log[i] + rng.uniform(-noise, noise)
    for i in [4, 5]:
        scale = max(abs(true_log[i]), 0.05)
        noisy[i] = true_log[i] + rng.uniform(-2.0 * scale, 2.0 * scale)
    return noisy


def fit_one_model(model_name, reg_map_ord, nns_grid, ordered_grid_coords_np, initial_vals):
    params = [torch.tensor([val], device=DEVICE, dtype=DTYPE, requires_grad=True) for val in initial_vals]
    if model_name == "Vecc_Std":
        model = kernels_vecchia.fit_vecchia_lbfgs(
            smooth=v, input_map=reg_map_ord, nns_map=nns_grid,
            mm_cond_number=mm_cond_number, nheads=nheads,
            limit_A=limit_a, limit_B=limit_b + 1, limit_C=limit_c + 1,
            daily_stride=daily_stride,
        )
    elif model_name == "Vecc_Advec":
        model = kernels_vecchia_advec.fit_vecchia_lbfgs_advec(
            smooth=v, input_map=reg_map_ord, nns_map=nns_grid,
            mm_cond_number=mm_cond_number, nheads=nheads,
            limit_A=limit_a, limit_B=limit_b, limit_C=limit_c,
            daily_stride=daily_stride,
            spatial_coords=ordered_grid_coords_np,
            advec_lon_offset=advec_lon_offset,
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

    model.precompute_conditioning_sets()
    optimizer = model.set_optimizer(params, lr=1.0, max_iter=lbfgs_eval, history_size=lbfgs_hist)
    start = time.time()
    out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=lbfgs_steps, grad_tol=1e-5)
    elapsed = time.time() - start
    return out, float(out[-1]), int(n_iter), elapsed


In [ ]:
def build_summary(records):
    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    overall = df.groupby("model").agg(
        completed=("iter", "nunique"),
        mean_rmsre=("rmsre", "mean"),
        median_rmsre=("rmsre", "median"),
        sd_rmsre=("rmsre", "std"),
        mean_loss=("loss", "mean"),
        mean_time_s=("time_s", "mean"),
        p90_p10_rmsre=("rmsre", lambda x: float(np.percentile(x, 90) - np.percentile(x, 10))),
    ).reset_index()

    rows = []
    true_vals = [true_dict[k] for k in P_LABELS]
    for label, col, tv in zip(P_LABELS, P_COLS, true_vals):
        row = {"parameter": label, "true": tv}
        for model in MODELS:
            vals = df.loc[df["model"] == model, col].dropna().to_numpy(dtype=float)
            if vals.size == 0:
                continue
            err = vals - tv
            row[f"{model}_mean"] = float(np.mean(vals))
            row[f"{model}_median"] = float(np.median(vals))
            row[f"{model}_sd"] = float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0
            row[f"{model}_p10"] = float(np.percentile(vals, 10))
            row[f"{model}_p90"] = float(np.percentile(vals, 90))
            row[f"{model}_p90_p10"] = float(np.percentile(vals, 90) - np.percentile(vals, 10))
            row[f"{model}_bias"] = float(np.mean(err))
            row[f"{model}_mae"] = float(np.mean(np.abs(err)))
            if abs(tv) >= 0.01:
                row[f"{model}_rmsre"] = float(np.sqrt(np.mean(((vals - tv) / abs(tv)) ** 2)))
            else:
                row[f"{model}_rmse"] = float(np.sqrt(np.mean(err ** 2)))
        rows.append(row)

    by_param = pd.DataFrame(rows)
    return overall, by_param


def print_running_summary(records):
    overall, by_param = build_summary(records)
    if overall.empty:
        print("No completed records yet.")
        return
    display(overall)
    display(by_param)


## Setup Shared Grid and Ordering

The maxmin ordering and neighbor map are computed once and shared by both models and all iterations.


In [ ]:
true_log = true_to_log_params(true_dict)
true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)

print("Building target grid...")
lats_grid, lons_grid, grid_coords = build_target_grid(lat_range, lon_range)
n_lat, n_lon = len(lats_grid), len(lons_grid)
n_grid = grid_coords.shape[0]
print(f"Grid: {n_lat} lat x {n_lon} lon x {T_STEPS} time = {n_grid * T_STEPS:,} rows")

print("Computing shared maxmin ordering...")
ord_grid, nns_grid = compute_grid_ordering(grid_coords, mm_cond_number)
ordered_grid_coords_np = grid_coords[ord_grid].detach().cpu().numpy()
print(f"Ordering done: N_grid={n_grid}, mm_cond_number={mm_cond_number}")


## Run Simulation

Both models use the same simulated field and the same initialization in each iteration. Results are checkpointed after every iteration.


In [ ]:
records = []
skipped = 0

for it in range(num_iters):
    print("\n" + "=" * 70)
    print(f"Iteration {it + 1}/{num_iters}  skipped={skipped}")
    print("=" * 70)

    initial_vals = make_random_init(rng, true_log, init_noise)
    init_orig = backmap_params(initial_vals)
    print(f"Init sigmasq={init_orig['sigmasq']:.3f} range_lon={init_orig['range_lon']:.3f} "
          f"advec_lat={init_orig['advec_lat']:.3f} advec_lon={init_orig['advec_lon']:.3f} "
          f"nugget={init_orig['nugget']:.3f}")

    try:
        field = generate_field_values(n_lat, n_lon, T_STEPS, true_params)
        reg_map = assemble_reg_map(field, grid_coords, true_params)
        del field
        reg_map_ord = {k: v[ord_grid] for k, v in reg_map.items()}

        for model_name in MODELS:
            print(f"\n--- {model_name} ---")
            out, loss, n_fit_iter, elapsed = fit_one_model(
                model_name, reg_map_ord, nns_grid, ordered_grid_coords_np, initial_vals,
            )
            rmsre, mae_zero, est = calculate_rmsre(out, true_dict)
            row = {
                "iter": it + 1,
                "model": model_name,
                "rmsre": rmsre,
                "mae_zero_true": mae_zero,
                "loss": loss,
                "fit_iter": n_fit_iter,
                "time_s": elapsed,
                "sigmasq_est": est["sigmasq"],
                "range_lat_est": est["range_lat"],
                "range_lon_est": est["range_lon"],
                "range_t_est": est["range_time"],
                "advec_lat_est": est["advec_lat"],
                "advec_lon_est": est["advec_lon"],
                "nugget_est": est["nugget"],
                "init_sigmasq": init_orig["sigmasq"],
                "init_range_lon": init_orig["range_lon"],
                "init_advec_lat": init_orig["advec_lat"],
                "init_advec_lon": init_orig["advec_lon"],
                "init_nugget": init_orig["nugget"],
            }
            records.append(row)
            print(f"loss={loss:.6f}  RMSRE={rmsre:.4f}  time={elapsed:.1f}s")

    except Exception as exc:
        skipped += 1
        import traceback
        print(f"[SKIP] Iteration {it + 1}: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        continue

    df_live = pd.DataFrame(records)
    df_live.to_csv(checkpoint_file, index=False)
    print(f"Checkpoint saved: {checkpoint_file}")
    print_running_summary(records)

print(f"Done: completed={len(pd.DataFrame(records)['iter'].unique()) if records else 0}, skipped={skipped}")


## Final Summary and Save

`p90_p10` columns report the central 80% spread of estimates for each parameter and model.


In [ ]:
df_results = pd.DataFrame(records)
if df_results.empty:
    print("No completed fits.")
else:
    overall_summary, param_summary = build_summary(records)

    display(df_results)
    display(overall_summary)
    display(param_summary)

    df_results.to_csv(raw_file, index=False)
    overall_summary.to_csv(summary_file.with_name(summary_file.stem + "_overall.csv"), index=False)
    param_summary.to_csv(summary_file, index=False)

    print(f"Saved raw: {raw_file}")
    print(f"Saved overall summary: {summary_file.with_name(summary_file.stem + '_overall.csv')}")
    print(f"Saved parameter summary: {summary_file}")
